# POI Ingestion - Bronze Layer

Downloads Geofabrik OSM extracts for all training states, parses each with a combined handler
that extracts both **branded POIs** (partners/competitors by name) and **general POI categories**
(retail, food_drink, etc. by OSM tag) in a single pass per PBF file.

**Data Source:** Geofabrik OSM PBF extracts → parsed with osmium

**Outputs:**
- `{catalog}.{bronze_schema}.raw_pois` — Branded partner/competitor POIs (expansion_state only)
- `{catalog}.{bronze_schema}.osm_pois_raw` — General POI categories for all training states

**Brand Configuration:** Loaded from `poi_config.yml`
**Category Mappings:** Loaded from `osm_poi_categories.yml`

## Parameters

In [ ]:
import requests
import time
import yaml
import os
import shutil
import osmium
from pyspark.sql import functions as F
from pyspark.sql.types import *
from collections import Counter

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("config_path", "")
dbutils.widgets.text("expansion_state", "MA")
dbutils.widgets.text("state_filter", "MA,MI,VA,NY,WA,MD,NJ")
dbutils.widgets.text("osm_categories_config", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
config_path = dbutils.widgets.get("config_path")
expansion_state = dbutils.widgets.get("expansion_state")
state_filter = dbutils.widgets.get("state_filter")
osm_categories_config = dbutils.widgets.get("osm_categories_config")

assert catalog and bronze_schema and config_path, "Missing required parameters: catalog, bronze_schema, config_path"

# Parse state filter
state_list = [s.strip() for s in state_filter.split(",") if s.strip()]
assert len(state_list) > 0, "At least one state must be specified in state_filter"

branded_output_table = f"{catalog}.{bronze_schema}.raw_pois"
general_output_table = f"{catalog}.{bronze_schema}.osm_pois_raw"

# OSM extract URLs for all supported states
OSM_EXTRACTS = {
    "MA": "https://download.geofabrik.de/north-america/us/massachusetts-latest.osm.pbf",
    "MI": "https://download.geofabrik.de/north-america/us/michigan-latest.osm.pbf",
    "VA": "https://download.geofabrik.de/north-america/us/virginia-latest.osm.pbf",
    "NY": "https://download.geofabrik.de/north-america/us/new-york-latest.osm.pbf",
    "WA": "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf",
    "MD": "https://download.geofabrik.de/north-america/us/maryland-latest.osm.pbf",
    "NJ": "https://download.geofabrik.de/north-america/us/new-jersey-latest.osm.pbf",
    "CT": "https://download.geofabrik.de/north-america/us/connecticut-latest.osm.pbf",
}

# Validate all states have URLs
for state in state_list:
    if state not in OSM_EXTRACTS:
        raise ValueError(f"No OSM extract URL for state: {state}")

print(f"Catalog: {catalog}")
print(f"Schema: {bronze_schema}")
print(f"Expansion state (branded POIs): {expansion_state}")
print(f"Training states (general POIs): {state_list}")
print(f"Branded output: {branded_output_table}")
print(f"General output: {general_output_table}")

## Load Brand and Category Configuration

In [ ]:
# Load brand configuration from poi_config.yml (single source of truth)
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

brands_config = config.get('brands', {})
partner_brands = brands_config.get('partner_brands', [])
competitor_brands = brands_config.get('competitor_brands', [])

assert partner_brands, "No partner_brands defined in poi_config.yml"
assert competitor_brands, "No competitor_brands defined in poi_config.yml"

all_brands = partner_brands + competitor_brands

print(f"Partner brands ({len(partner_brands)}): {partner_brands}")
print(f"Competitor brands ({len(competitor_brands)}): {competitor_brands}")

# Load general POI category mappings from osm_poi_categories.yml
# Build a lookup: (osm_key, osm_value) → category_name
category_lookup = {}  # {(key, value): category}
if osm_categories_config:
    with open(osm_categories_config, 'r') as f:
        cat_config = yaml.safe_load(f)
    
    for category_name, tag_groups in cat_config.get('poi_categories', {}).items():
        for osm_key, osm_values in tag_groups.items():
            for val in osm_values:
                category_lookup[(osm_key, val)] = category_name
    
    print(f"\nGeneral POI categories: {list(cat_config.get('poi_categories', {}).keys())}")
    print(f"Total tag→category mappings: {len(category_lookup)}")
else:
    print("\nWARNING: No osm_categories_config provided, skipping general POI extraction")

## Download & Parse OSM Extracts (All Training States)

In [ ]:
# Download OSM extracts for all training states to Databricks volume
osm_volume_path = f"/Volumes/{catalog}/{bronze_schema}/osm_data/"

# Ensure volume directory exists
try:
    dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_schema}/osm_data/")
except:
    pass

state_file_paths = {}
for state in state_list:
    osm_url = OSM_EXTRACTS[state]
    osm_filename = osm_url.split('/')[-1]
    osm_file_path = f"{osm_volume_path}{osm_filename}"
    
    if os.path.exists(osm_file_path):
        file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
        print(f"  {state}: cached ({file_size_mb:.1f} MB)")
    else:
        print(f"  {state}: downloading from {osm_url}...")
        start_time = time.time()
        with requests.get(osm_url, stream=True, timeout=600,
                         headers={"User-Agent": "DatabricksGeospatialPipeline/1.0"}) as r:
            r.raise_for_status()
            with open(osm_file_path, "wb") as f:
                shutil.copyfileobj(r.raw, f, length=16*1024*1024)
        dl_time = time.time() - start_time
        file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
        print(f"  {state}: downloaded {file_size_mb:.1f} MB in {dl_time:.1f}s")
    
    state_file_paths[state] = osm_file_path

print(f"\n{'='*60}")
print(f"OSM extracts ready for {len(state_file_paths)} states")
print(f"{'='*60}")

In [ ]:
# Combined handler: extracts branded POIs (by name) and general POI categories (by tag) in one pass

class CombinedPOIHandler(osmium.SimpleHandler):
    """OSM handler that extracts branded POIs and general category POIs simultaneously."""
    
    def __init__(self, brand_patterns, category_lookup):
        super().__init__()
        self.branded_pois = []
        self.general_pois = []
        self.brand_patterns = [b.lower() for b in brand_patterns]
        self.category_lookup = category_lookup  # {(key, value): category}
        # OSM keys to check for brands
        self.brand_tag_keys = ['amenity', 'shop']
        # OSM keys to check for categories
        self.category_keys = set(k for k, v in category_lookup.keys())
    
    def _matches_brand(self, tags):
        name = tags.get('name', '').lower()
        if not name:
            return False
        return any(pattern in name for pattern in self.brand_patterns)
    
    def _classify_category(self, tags):
        for key in self.category_keys:
            val = tags.get(key)
            if val:
                cat = self.category_lookup.get((key, val))
                if cat:
                    return cat
        return None
    
    def _has_poi_tag(self, tags):
        return any(key in tags for key in self.brand_tag_keys)
    
    def _process_element(self, osm_id, osm_type, lat, lon, tags):
        # General category classification (by tag)
        category = self._classify_category(tags)
        if category:
            self.general_pois.append({
                'osm_id': str(osm_id),
                'osm_type': osm_type,
                'latitude': lat,
                'longitude': lon,
                'poi_category': category,
            })
        
        # Branded name match
        if self._matches_brand(tags):
            self.branded_pois.append({
                'osm_id': str(osm_id),
                'osm_type': osm_type,
                'latitude': lat,
                'longitude': lon,
                'tags': dict(tags),
            })
    
    def node(self, n):
        if n.location.valid():
            tags = dict(n.tags)
            if tags:
                self._process_element(n.id, 'node', n.location.lat, n.location.lon, tags)
    
    def way(self, w):
        tags = dict(w.tags)
        if not tags:
            return
        lats, lons = [], []
        for node in w.nodes:
            if node.location.valid():
                lats.append(node.location.lat)
                lons.append(node.location.lon)
        if lats and lons:
            self._process_element(w.id, 'way', sum(lats)/len(lats), sum(lons)/len(lons), tags)


# Parse all training states
all_branded = []
all_general = []

for state in state_list:
    osm_file_path = state_file_paths[state]
    print(f"\nParsing {state}: {osm_file_path}")
    
    start_time = time.time()
    handler = CombinedPOIHandler(brand_patterns=all_brands, category_lookup=category_lookup)
    handler.apply_file(osm_file_path, locations=True)
    parse_time = time.time() - start_time
    
    # Tag with state
    for poi in handler.general_pois:
        poi['state'] = state
    for poi in handler.branded_pois:
        poi['state'] = state
    
    all_general.extend(handler.general_pois)
    all_branded.extend(handler.branded_pois)
    
    print(f"  {state}: {len(handler.general_pois):,} general POIs, {len(handler.branded_pois)} branded POIs in {parse_time:.1f}s")

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_general):,} general POIs, {len(all_branded)} branded POIs across {len(state_list)} states")
print(f"{'='*60}")

# Show general POI category distribution
cat_counts = Counter(p['poi_category'] for p in all_general)
print("\nGeneral POI categories:")
for cat, count in cat_counts.most_common():
    print(f"  {cat}: {count:,}")

# Show branded POI distribution
brand_counts = Counter(p['tags'].get('name', 'Unknown') for p in all_branded)
print(f"\nBranded POIs:")
for brand, count in brand_counts.most_common(20):
    print(f"  {brand}: {count}")

## Write to Bronze Tables

In [ ]:
# ---- 1. Write branded POIs (expansion_state only, same schema as before) ----
branded_expansion = [p for p in all_branded if p.get('state') == expansion_state]
print(f"Branded POIs for {expansion_state}: {len(branded_expansion)}")

if branded_expansion:
    branded_schema = StructType([
        StructField("osm_id", StringType(), False),
        StructField("osm_type", StringType(), False),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("tags", MapType(StringType(), StringType()), True)
    ])
    branded_df = spark.createDataFrame(branded_expansion, schema=branded_schema)
    branded_df = branded_df.withColumn("ingestion_timestamp", F.current_timestamp())

    (branded_df
     .write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(branded_output_table))
    print(f"Written {branded_df.count()} branded POIs to {branded_output_table}")
else:
    print(f"WARNING: No branded POIs found for {expansion_state}")

# ---- 2. Write general POIs (all training states) ----
if all_general:
    general_schema = StructType([
        StructField("osm_id", StringType(), False),
        StructField("osm_type", StringType(), False),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("poi_category", StringType(), False),
        StructField("state", StringType(), False),
    ])
    general_df = spark.createDataFrame(all_general, schema=general_schema)
    general_df = general_df.withColumn("ingestion_timestamp", F.current_timestamp())

    (general_df
     .write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(general_output_table))
    print(f"Written {general_df.count():,} general POIs to {general_output_table}")
else:
    print("WARNING: No general POIs extracted")

## Validation

In [ ]:
print("=" * 80)
print("POI INGESTION VALIDATION")
print("=" * 80)

# Validate branded POIs (expansion_state)
print(f"\n1. Branded POIs Table ({branded_output_table}):")
try:
    branded_summary = spark.sql(f"""
        SELECT 
            COUNT(*) as total_pois,
            COUNT(DISTINCT osm_id) as unique_pois,
            COUNT(CASE WHEN tags['name'] IS NOT NULL THEN 1 END) as pois_with_name
        FROM {branded_output_table}
    """)
    display(branded_summary)
    
    print("\nBranded POIs by name:")
    display(spark.sql(f"""
        SELECT tags['name'] as name, osm_type, COUNT(*) as count
        FROM {branded_output_table}
        WHERE tags['name'] IS NOT NULL
        GROUP BY tags['name'], osm_type
        ORDER BY count DESC
    """))
except Exception as e:
    print(f"  ERROR: {e}")

# Validate general POIs (all training states)
print(f"\n2. General POIs Table ({general_output_table}):")
try:
    general_summary = spark.sql(f"""
        SELECT 
            COUNT(*) as total_pois,
            COUNT(DISTINCT poi_category) as categories,
            COUNT(DISTINCT state) as states
        FROM {general_output_table}
    """)
    display(general_summary)
    
    print("\nGeneral POIs by state and category:")
    display(spark.sql(f"""
        SELECT state, poi_category, COUNT(*) as count
        FROM {general_output_table}
        GROUP BY state, poi_category
        ORDER BY state, count DESC
    """))
except Exception as e:
    print(f"  ERROR: {e}")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)